In [50]:
%run "./log_monitoring"

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 70, Finished, Available, Finished, True)

In [51]:
from pyspark.sql.functions import col, udf, lit, when, monotonically_increasing_id,\
    input_file_name, regexp_extract, to_timestamp, current_timestamp,\
    col, lit, isnan, collect_list, concat_ws, col, max
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DataType, TimestampType
from pyspark.sql.types import BooleanType
import re
import uuid

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 71, Finished, Available, Finished, False)

In [52]:
system='boafa'
table_name='transaction'
watermarkColumn='LastModifiedDate'
watermarkValue='1900-01-01'
loadType='Incremental'
keyColumn='TRANSACTION_ID,TRANSACTION_LINE_ID'
lakehouse='BOAFALakehouse'
source_schema='clean'
target_schema='prepare'
config_schema='config'
config_table='metadata'
monitoring_table='pipeline_monitoring'

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 72, Finished, Available, Finished, False)

In [53]:
cleanView=f"{lakehouse}.{source_schema}.view_{system}_{table_name}"
preparetable=f"{lakehouse}.{target_schema}.{system}_{table_name}"
quarantine_prepare=f"{lakehouse}.{target_schema}.{system}_quarantine_{table_name}"
monitoring_table_fullname=f"{lakehouse}.{config_schema}.{monitoring_table}"

spark.conf.set("cleanView", cleanView)
tempView=f"view_{system}_{table_name}_temp"

spark.conf.set("tempView", tempView)
spark.conf.set("loadType", loadType)
if loadType=="Full":
    spark.conf.set("watermarkValue", ' ')
    spark.conf.set("watermarkColumn", ' ')
else:
    spark.conf.set("watermarkValue", watermarkValue)
    spark.conf.set("watermarkColumn", watermarkColumn)

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 73, Finished, Available, Finished, False)

In [54]:
import sempy.fabric as fabric
workspace_name = None
wsid = fabric.get_workspace_id() if workspace_name is None else fabric.resolve_workspace_id(workspace_name)

lakehouse_name = lakehouse
spark.conf.set("ws_id", wsid)
spark.conf.set("lh_name",lakehouse_name)

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 74, Finished, Available, Finished, False)

In [56]:
%%spark
import com.microsoft.spark.fabric.tds.implicits.read.FabricSparkTDSImplicits._
import com.microsoft.spark.fabric.Constants

val scala_cleanView = spark.conf.get("cleanView")
val scala_loadType = spark.conf.get("loadType")
val watermarkColumn = spark.conf.get("watermarkColumn")
val watermarkValue = spark.conf.get("watermarkValue")

val t_sql_query = if (scala_loadType == "Full") {
    s"SELECT * FROM $scala_cleanView"
} else {
    s"SELECT * FROM $scala_cleanView WHERE $watermarkColumn > '$watermarkValue'"
}

val wsid = spark.conf.get("ws_id")
val lh_name = spark.conf.get("lh_name")

val scala_df = spark.read
    .option(Constants.WorkspaceId, wsid)
    .option(Constants.DatabaseName, lh_name)
    .synapsesql(t_sql_query)

val scala_tempView = spark.conf.get("tempView")
scala_df.createOrReplaceTepmView(scala_tempView)


StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, 77, Finished, Available, Finished, False)

Request to read failed. Error - Failure to render dataframe.; Additional Detail - , Cause - Invalid object name 'BOAFALakehouse.clean.view_boafa_transaction'.


Error: com.microsoft.spark.fabric.tds.read.error.FabricSparkTDSReadError: Failure to render dataframe.

In [ ]:
def generate_merge():
    schema_info=spark.sql(f"DESCRIBE {rawtable}")

    #sourcequery=f"select * from {rawtable} where {watermarkColumn}>'{watermarkValue}'"
    sourceview=f"view_{system}_{table_name}_temp"

    key_column_array=keyColumn.split(",")
    join_condition = " AND ".join([f"Target.{col}" + f" = Source.{col}" for col in key_column_array])

    #### Merge Script #####
    update_clause=''
    insert_cloase_target=''
    insert_cloase_source='' 

    for row in schema_info.collect():
        if row[col_name] not in key_column_array:
            update_clause += f" Target.{row['col_name']} = Source.{row['col_name']},\n"
        
        insert_cloase_source += f"Source.{row['col_name']}, "
        insert_cloase_target += f"{row['col_name']}, "

    update_clause=update_clause.rstrip(",\n")
    insert_cloase_source=insert_cloase_source.rstrip(", ")
    insert_cloase_target=insert_cloase_target.rstrip(", ")

    merge_script=''
    merge_script= f"merge into {cleantable} as Target \n \
                    using {sourceview} as Source \n \
                    on {join_condition}                 \n \
                    when matched then                   \n \
                    update set {update_clause}          \n \
                    when not matched then               \n \
                    insert ({insert_cloase_target}) values({insert_cloase_source});"

    spark.sql(f"""INSERT INTO {lakehouse}.{config_schema}.{config_table}(system,tier,table_name,key_name,value) 
                  VALUES('{system}','{target_schema}','{table_name}',"mergeScript",'{merge_script}')""")

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, -1, Cancelled, , Cancelled, True)

In [ ]:
def full_load():
    if loadType=="Full":
        source_raw_count=0
        target_raw_count=0
        dq_issue_count=0
        remarks='No new data to load'
        df=spark.sql(f"SELECT * FROM {tempView}")

        if not df.rdd.isEmpty():
            ######## run dq check #########
            df.write.formate("delta").mode("overwrite").saveAsTable(preparetable)
            
            source_raw_count=df.count()
            target_raw_count=source_raw_count
            remarks=f"Data loaded to {preparetable} table"
        
    return source_raw_count,target_raw_count,dq_issue_count,remarks

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, -1, Cancelled, , Cancelled, True)

In [ ]:
def incremental_load():
    if loadType=='Incremental':
        source_raw_count=0
        target_raw_count=0
        dq_issue_count=0
        remarks='No new data to load'

        sourceview=tempView
        
        df=spark.sql(f"SELECT * FROM {tempView}")

        if df.rdd.isEmpty():
            print("no new data to load")
        else:

            source_raw_count=df.count()
            target_raw_count=df.count()
            remarks=f"Data loaded to {preparetable} table"

            if spark.catalog.tableExists(preparetable):
                merge_script=spark.sql(f"""SELECT value from {lakehouse}.{config_schema}.{config_table}
                    where key_name='mergeScript' and table_name='{table_name}'
                    and tier='{target_schema}' and system='{system}'""").first()[0]
                print("merge start")
                spark.sql(merge_script)
                print("merge complete")

            else:
                df.write.formate("delta").saveAsTable(preparetable)
                print("initial load done, table created")
                spark.sql(f"""DELETE from {lakehouse}.{config_schema}.{config_table}
                            where key_name='merge_script' and table_name='{table_name}'
                            and tier='{target_schema}' and system='{system}'""")
                generate_merge()
                print("merge script generated")
        
            watermark_update()

    return  source_raw_count,target_raw_count,dq_issue_count,remarks

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, -1, Cancelled, , Cancelled, True)

In [ ]:
def watermark_update():
    if loadType=='Incremental':
        df=spark.table(preparetable)

        max_value=df.select(max(col(watermarkColumn)).first()[0])

        if max_value is None:
            max_value='1900-01-01'

        max_value_formated_str = max_value

        update_query=f"""update {lakehouse}.{config_schema}.{config_table}
                                set value='{max_value_formated_str}'
                                where table_name='{table_name}'
                                and key_name='watermarkValue'
                                and tier='{target_schema}'
                                and system={system}"""
        
        spark.sql(update_query)
        print("watermark table updated")

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, -1, Cancelled, , Cancelled, True)

In [ ]:
# if spark.catalog.tableExists(f"{lakehouse}.{source_schema}_{table_name}"):
#     try:
#         start_time = datetime.now()
#         pipeline_name="prepare_to_clean"
#         run_id= str(uuid.uuid4())
#         layer_from=source_schema
#         layer_to=target_schema

#         lod=start_pipeline_log(system,run_id,start_time,pipeline_name,layer_from,loyer_to,f"{system}_{table_name}")

#         if loadType=="Full":
#             source_raw_count,target_raw_count,dq_issue_count,remarks = full_load()

#         if loadType=="Incremental":
#             source_raw_count,target_raw_count,dq_issue_count,remarks = incremental_load()

#         end_pipeline_log(monitoring_table_fullname,log,"Success",source_raw_count,target_raw_count,dq_issue_count,remarks)

#     except Exception as e:
#         error_message=f"{e}"
#         source_raw_count=0
#         target_raw_count=0
#         end_pipeline_log(monitoring_table_fullname,log,"Fail",source_raw_count,target_raw_count,0,'',error_message)

StatementMeta(, d7526693-b986-4f9c-98dd-6d376848a547, -1, Cancelled, , Cancelled, True)